# Fake News & Source Credibility Detector — Colab GPU Runner

End-to-end training + evaluation on a Colab **GPU**.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or A100 for the Mistral LLM step).

Run the cells top to bottom. The heavy step is **Cell 7 (DeBERTa training)**, ~40–60 min on a free T4.

---
Pipeline: data → TF-IDF baseline → **DeBERTa (GPU)** → calibration → speaker profiles → SHAP → (optional) Mistral LLM → save weights.


## 1 · Verify the GPU


In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print('NO GPU — go to Runtime → Change runtime type → T4 GPU, then re-run.')


## 2 · Get the code

**Option A (recommended): clone from GitHub.** Always the latest version.

If you instead uploaded `colab_bundle.zip`, skip this cell and run the *Option B* cell below.


In [ ]:
# Option A — clone the public repo
import os
REPO = 'https://github.com/Sanjana132/FakeNewsCredibilityAssesor.git'
if not os.path.exists('/content/FakeNewsCredibilityAssesor'):
    !git clone $REPO /content/FakeNewsCredibilityAssesor
%cd /content/FakeNewsCredibilityAssesor
!ls


In [ ]:
# Option B — use an uploaded colab_bundle.zip instead of cloning.
# from google.colab import files; up = files.upload()   # pick colab_bundle.zip
# import zipfile; zipfile.ZipFile('colab_bundle.zip').extractall('/content/')
# %cd /content/colab_bundle
# !ls
print('Skip this cell if you cloned in Option A.')


## 3 · Install dependencies (training stack) + NLTK data


In [ ]:
# requirements.train.txt pulls in requirements.api.txt automatically.
!pip install -q -r requirements.train.txt
import nltk
for pkg in ['stopwords','punkt','punkt_tab','opinion_lexicon']:
    nltk.download(pkg, quiet=True)
print('Dependencies installed.')


## 4 · Phase 1–3 · Build the dataset (downloads LIAR-2, MultiFC, FEVER, AVeriTeC)

Skip this cell if `data/train.csv`, `data/val.csv`, `data/test.csv` already exist (e.g. you uploaded them).

Add `--sample 3000` for a fast smoke run; omit it for the full ~90k-row dataset.


In [ ]:
import os
if all(os.path.exists(f'data/{s}.csv') for s in ['train','val','test']):
    print('Data already present — skipping Phase 1–3.')
else:
    !python credibility_detector_phases123.py 2>&1 | tail -40


In [ ]:
# Sanity check the splits (expect 0 NaN in credibility_score)
import pandas as pd
for s in ['train','val','test']:
    df = pd.read_csv(f'data/{s}.csv')
    print(f'{s:5} {len(df):>7,} rows   NaN credibility={df["credibility_score"].isna().sum()}')


## 5 · Phase 4 · TF-IDF baseline (sets the MAE benchmark DeBERTa must beat)


In [ ]:
!python phase4_baseline_1.py --no-shap


## 6 · Phase 5 · Fine-tune DeBERTa-v3 on GPU  ⏳ (main step)

Expected on a T4: ~5–8 min/epoch, early-stopping usually by epoch 4–6. Target val MAE ≈ 0.20–0.24 (baseline ≈ 0.287).


In [ ]:
!python phase5_deberta.py --train --device cuda


In [ ]:
# Show the results
import json
r = json.load(open('models/deberta_results.json'))
print('Best epoch   :', r['best_epoch'])
print('Best val MAE :', r['best_val_MAE'])
print('Test MAE     :', r['test']['MAE'])
print('Test F1      :', r['test']['Macro_F1'])
print('Baseline MAE :', r['baseline_MAE'])
print('Improvement  :', r['improvement'])
print('Per-dataset  :', json.dumps(r.get('per_dataset', {}), indent=2))


## 7 · Phase 5b · Calibration & uncertainty

Reliability diagram, ECE, and MC-Dropout 90% CI coverage. Applies temperature scaling if ECE > 0.05.


In [ ]:
!python phase5b_calibration.py --device cuda


## 8 · Phase 6 · Speaker profiles + context encoder


In [ ]:
!python phase6_speaker_profiler.py
!python context_encoder.py --visualise


## 9 · Phase 7 · Token-level SHAP explanations

Produces `eda_output/12_deberta_shap.html` and a global-importance chart.


In [ ]:
!python phase7_shap_explainer.py --n-examples 12 --n-global 150


## 10 · (Optional) Phase 7 · Mistral-7B QLoRA justification model

**Needs a high-RAM / A100 runtime** (4-bit Mistral-7B). Skip on a plain T4 if you hit OOM.
Trains only on rows that have a justification (LIAR-2 + AVeriTeC).


In [ ]:
# Uncomment to run (A100 recommended):
# !python llm_finetune.py --train


## 11 · Save trained artifacts to Google Drive

So your weights survive the runtime disconnecting.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
dst = '/content/drive/MyDrive/FakeNews_models'
os.makedirs(dst, exist_ok=True)
for f in ['deberta_best.pt','deberta_results.json','deberta_tokenizer',
          'baseline_tfidf.pkl','baseline_results.json','calibration.json',
          'speaker_profiles.json']:
    src = f'models/{f}'
    if os.path.exists(src):
        d = f'{dst}/{f}'
        if os.path.isdir(src):
            shutil.rmtree(d, ignore_errors=True); shutil.copytree(src, d)
        else:
            shutil.copy(src, d)
        print('saved', f)
print('Done → Drive/MyDrive/FakeNews_models')


In [ ]:
# Alternative: download a single zip of everything to your machine
import shutil
shutil.make_archive('/content/trained_models', 'zip', 'models')
from google.colab import files
files.download('/content/trained_models.zip')


## 12 · (Optional) Quick inference smoke test

Confirms the saved model scores a claim end-to-end (DeBERTa + MC-Dropout CI + token SHAP).


In [ ]:
!python phase5_deberta.py --predict "The unemployment rate fell to a record low." \
    --speaker "Joe Biden" --context "a speech" --device cuda
